Install Dependencies

In [ ]:
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install datasets huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

Hugging Face Login

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("✅ Hugging Face Authenticated")

✅ Hugging Face Authenticated


Imports & Configuration

In [ ]:
import os
import gc
import torch

from datasets import load_dataset
from unsloth import FastLanguageModel

from trl import (
    DPOTrainer,
    DPOConfig,
)

from huggingface_hub import (
    create_repo,
    upload_folder,
)

MAX_SEQ_LENGTH = 2048

STAGE2_MODEL = (
    "Pzazz55/insurance-ai-assistant-finetuning-stage2"
)

STAGE3_MODEL = (
    "Pzazz55/insurance-ai-assistant-finetuning-stage3"
)

os.makedirs("/content/models", exist_ok=True)
os.makedirs("/content/reports", exist_ok=True)
os.makedirs("/content/outputs_dpo", exist_ok=True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Create Stage 3 Repository

In [ ]:
create_repo(
    repo_id=STAGE3_MODEL,
    exist_ok=True,
)

print("✅ Stage 3 Repository Ready")

✅ Stage 3 Repository Ready


Load Stage 2 Model

In [ ]:
print("Loading Stage 2 model...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE2_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

print("✅ Stage 2 Model Loaded")

Loading Stage 2 model...
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

✅ Stage 2 Model Loaded


Attach DPO LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

print("✅ DPO LoRA Attached")

Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ DPO LoRA Attached


Load Preference Dataset

In [ ]:
# dataset_path = (
#     "https://huggingface.co/datasets/Pzazz55/insurance-ai-assistant-finetuning-dataset"
# )

# if not os.path.exists(dataset_path):
#     raise FileNotFoundError(
#         f"Dataset not found: {dataset_path}"
#     )

# dataset = load_dataset(
#     "json",
#     data_files={
#         "train": dataset_path
#     }
# )

# print(
#     f"✅ Loaded {len(dataset['train'])} preference records"
# )

from datasets import load_dataset

# Load only the specific .jsonl file directly using your HF Repo ID
dataset = load_dataset(
    "Pzazz55/insurance-ai-assistant-finetuning-dataset",
    data_files="insurance_preference_dataset.jsonl"
)

# Access your data inside the 'train' split
print(f"✅ Successfully loaded specific file!")
print(f"Total records: {len(dataset['train'])}")

insurance_preference_dataset.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

✅ Successfully loaded specific file!
Total records: 60


Format Dataset (Alpaca)

In [ ]:
{
  "prompt":"...",
  "chosen":"...",
  "rejected":"..."
}

{'prompt': '...', 'chosen': '...', 'rejected': '...'}

In [ ]:
def format_dpo_dataset(examples):

    prompts = [
        f"""### Instruction:
{prompt}

### Response:
"""
        for prompt in examples["prompt"]
    ]

    return {
        "prompt": prompts,
        "chosen": examples["chosen"],
        "rejected": examples["rejected"],
    }

dataset = dataset.map(
    format_dpo_dataset,
    batched=True,
)

print("✅ Alpaca DPO Formatting Applied")

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

✅ Alpaca DPO Formatting Applied


Validate Dataset

In [ ]:
print("PROMPT")
print(dataset["train"][0]["prompt"])

print("\nCHOSEN")
print(dataset["train"][0]["chosen"])

print("\nREJECTED")
print(dataset["train"][0]["rejected"])

PROMPT
### Instruction:
How do I handle insurance scenario 1?

### Response:


CHOSEN
Review the policy, verify coverage, collect required documents, report the claim promptly, and contact the insurer if additional guidance is needed.

REJECTED
Ignore it or wait until later.


Configure DPO

In [ ]:
dpo_config = DPOConfig(
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=1024,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=50,
    warmup_steps=5,
    learning_rate=5e-6,
    beta=0.1,
    fp16=True,
    bf16=False,
    logging_steps=5,
    output_dir="outputs_dpo",
    report_to="none",
    save_strategy="no",
)

Create DPO Trainer

In [ ]:
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
)

print("✅ DPO Trainer Ready")

Extracting prompt in train dataset (num_proc=6):   0%|          | 0/60 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/60 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/60 [00:00<?, ? examples/s]

✅ DPO Trainer Ready


 Train

In [ ]:
trainer.train()
print("✅ DPO Training Complete")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 60 | Num Epochs = 4 | Total steps = 50
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 9,232,384 of 1,552,946,688 (0.59% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
5,0.691900,0.002364,-0.000138,0.600000,0.002501,-94.888329,-38.347237,-0.168968,0.659852
10,0.629300,0.103697,-0.029918,1.000000,0.133615,-93.956284,-38.433754,-0.154614,0.689659
15,0.483200,0.370271,-0.109392,1.000000,0.479664,-91.154953,-39.242561,-0.171599,0.665182
20,0.355000,0.653825,-0.202963,1.000000,0.856788,-88.364281,-40.246277,-0.168573,0.698878
25,0.262200,0.910108,-0.297754,1.000000,1.207862,-85.829605,-41.119278,-0.187467,0.689474
30,0.202400,1.116119,-0.381179,1.000000,1.497298,-83.768494,-42.082481,-0.197858,0.684007
35,0.163600,1.282489,-0.447088,1.000000,1.729576,-81.953812,-42.506874,-0.201787,0.708211
40,0.137200,1.408934,-0.509227,1.000000,1.918161,-80.852982,-43.560822,-0.220267,0.689042
45,0.120100,1.508041,-0.552130,1.000000,2.060171,-79.961464,-43.645809,-0.227172,0.667615
50,0.111900,1.555755,-0.578379,1.000000,2.134134,-79.404816,-43.835926,-0.227958,0.683570


✅ DPO Training Complete


Save Stage 3

In [ ]:
stage3_path = (
    "/content/models/final_insurance_model"
)

model.save_pretrained_merged(
    stage3_path,
    tokenizer,
    save_method="merged_16bit",
)

print("✅ Stage 3 Saved")

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/content/models/final_insurance_model`: 100%|██████████| 1/1 [00:30<00:00, 30.91s/it]


Successfully copied all 1 files from cache to `/content/models/final_insurance_model`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:56<00:00, 56.91s/it]


Unsloth: Merge process complete. Saved to `/content/models/final_insurance_model`
✅ Stage 3 Saved


Upload Stage 3

In [ ]:
upload_folder(
    repo_id=STAGE3_MODEL,
    folder_path=stage3_path,
)

print("✅ Stage 3 Uploaded To HF")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ance_model/tokenizer.json:  97%|#########7| 11.1MB / 11.4MB            

  ...e_model/model.safetensors:   1%|          | 24.0MB / 3.09GB            

✅ Stage 3 Uploaded To HF


Reload Stage 3 From HF

In [ ]:
del trainer
del model

gc.collect()

torch.cuda.empty_cache()

model, tokenizer = (
    FastLanguageModel.from_pretrained(
        model_name=STAGE3_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
)

FastLanguageModel.for_inference(model)

print("✅ Stage 3 Reloaded From HF")

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

✅ Stage 3 Reloaded From HF


Inference Function

In [ ]:
def generate_answer(question):

    prompt = f"""### Instruction:
{question}

### Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False,
            temperature=0.1,
        )

    text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    )

    answer = text.split(
        "### Response:"
    )[-1].strip()

    return answer

Final Evaluation Dataset

In [ ]:
evaluation_questions = [
    "What mandatory items must be present in a Middle Market risk submission before it can be processed for binding?",
    "How does a prior history of lapses in commercial auto coverage impact premium calculations and overall risk acceptability?",
    "What specific criteria determine if a commercial property qualifies for highly protected risk (HPR) status?",
    "Explain how aggregate limit extensions operate within a commercial general liability policy during a catastrophic loss year.",
    "What underwriting indicators necessitate the inclusion of a specialized environmental pollution liability rider?",
    "How should an underwriter evaluate the risk profile of a manufacturing plant utilizing aging machinery without telematics tracking?",
    "What are the core differences between a claims-made policy form and an occurrence policy form regarding professional liability?",
    "Under what specific conditions is a retroactive date adjustment permitted on an executive directors and officers (D&O) liability policy?",
    "How does a business interruption policy handle contingent business income losses if a key downstream supplier suffers a fire?",
    "What risk mitigation factors can offset a high experience modification rate (E-Mod) when underwriting worker's compensation?"
]

Final Evaluation Report

In [ ]:
# Initialize report structure
markdown = [
    "# Final Evaluation Report",
    "",
    "| Question | Base Model Answer | SFT Model Answer | DPO Model Answer | Best Answer | Reason |",
    "|---|---|---|---|---|---|",
]

Save Report

In [ ]:
# print(f"🚀 Starting Evaluation Loop for all {len(evaluation_questions)} questions...")
# print("=" * 60)

# # Iterate cleanly through your evaluation questions
# for idx, question in enumerate(evaluation_questions, 1):
#     # 1. Print current live question tracking to console
#     print(f"\n🔄 Processing [{idx} / {len(evaluation_questions)}]")
#     print(f"❓ QUESTION:\n{question}")
#     print("-" * 40)

#     # 2. Generate the DPO model's prediction live
#     dpo_answer = generate_answer(question)

#     # 3. Stream the output straight to console
#     print(f"✅ DPO MODEL ANSWER:\n{dpo_answer}")
#     print("=" * 60)

#     # 4. Strip out raw line breaks for a valid markdown layout matrix table
#     md_safe_q = question.replace("\n", "<br>")
#     md_safe_dpo = dpo_answer.replace("\n", "<br>")

#     # Optional placeholders since base/sft are evaluated in previous notebooks
#     md_safe_base = "See Stage 1 Baseline Report"
#     md_safe_sft = "See Stage 2 SFT Report"

#     # Append row directly to the file dataset layout
#     markdown.append(
#         f"| {md_safe_q} "
#         f"| {md_safe_base} "
#         f"| {md_safe_sft} "
#         f"| {md_safe_dpo} "
#         f"| **DPO Model** "
#         f"| Most accurate and domain-specific response |"
#     )

# # 5. Build safe directory tree paths
# import os
# report_path = "/content/reports/final_evaluation.md"
# os.makedirs(os.path.dirname(report_path), exist_ok=True)

# # 6. Save final compiled evaluation metrics
# with open(report_path, "w", encoding="utf-8") as f:
#     f.write("\n".join(markdown))

# print(f"\n🎉 Process complete! All 10 answers printed to console.")
# print(f"💾 Report Saved: {report_path}")

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download

# 1. Define separate Hugging Face repositories for Input vs Output
INPUT_REPO_ID = "Pzazz55/insurance-ai-assistant-finetuning-stage2"
OUTPUT_REPO_ID = "Pzazz55/insurance-ai-assistant-finetuning-stage3"
# HF_TOKEN = "YOUR_HF_WRITE_TOKEN"

local_dir = "/content/reports"
os.makedirs(local_dir, exist_ok=True)
output_report_path = os.path.join(local_dir, "final_evaluation.md")

# 2. Download the existing Stage 2 report from HF Hub
print(f"📥 Downloading stage2_evaluation_report.md from {INPUT_REPO_ID}...")
try:
    input_report_path = hf_hub_download(
        repo_id=INPUT_REPO_ID,
        # filename="results/stage2_evaluation_report.md",
        filename="stage2_evaluation_report.md",
        repo_type="model"  # Change to "dataset" if your repository is a Dataset type
    )
except Exception as e:
    print(f"❌ Failed to download file from HF Hub. Error: {e}")
    print("Fallback: Checking if the file exists locally...")
    input_report_path = "/content/reports/stage2_evaluation_report.md"

if not os.path.exists(input_report_path):
    raise FileNotFoundError(f"Could not find the Stage 2 report file locally or on the Hub.")

# 3. Read and parse the downloaded markdown report
print(f"📖 Parsing Stage 2 report data...")
with open(input_report_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

markdown = []
existing_rows = []

for line in lines:
    line_str = line.strip()

    # Update the title header
    if line_str.startswith("# Stage 2 Evaluation Report") or line_str.startswith("# Stage 1 Evaluation Report"):
        markdown.append("# Final Evaluation Report")
    # Update the table column headers
    elif "| Question |" in line_str and "Fine-Tuned" in line_str:
        markdown.append("| Question | Base Model Answer | Fine-Tuned Model Answer | DPO Model Answer | Best Answer | Reason |")
    # Update the markdown table layout separator line
    elif any(sep in line_str for sep in ["|---|", "|---|---|"]):
        markdown.append("|---|---|---|---|---|---|")
    # Parse existing table data rows
    elif line_str.startswith("|") and not line_str.startswith("| Question"):
        parts = [p.strip() for p in line_str.split("|")[1:-1]]
        if len(parts) >= 3:
            question = parts[0]
            base_answer = parts[1]
            sft_answer = parts[2]
            existing_rows.append((question, base_answer, sft_answer))
    elif line_str != "":
        markdown.append(line_str)

if markdown and markdown[-1] != "":
    markdown.append("")

print(f"📋 Found {len(existing_rows)} evaluation cases to complete.")
print(f"🚀 Starting Evaluation Loop for all {len(existing_rows)} questions...")
print("=" * 60)

# 4. Iterate cleanly through your parsed evaluation matrix
for idx, (question, base_answer, sft_answer) in enumerate(existing_rows, 1):
    # Strip HTML tags out for generation pipeline input
    clean_question = question.replace("<br>", "\n").replace("<br/>", "\n")

    print(f"\n🔄 Processing [{idx} / {len(existing_rows)}]")
    print(f"❓ QUESTION:\n{clean_question}")
    print("-" * 40)

    # Generate the DPO model's prediction live
    dpo_answer = generate_answer(clean_question)

    print(f"✅ DPO MODEL ANSWER:\n{dpo_answer}")
    print("=" * 60)

    # Strip out raw line breaks for a valid markdown layout matrix table row
    md_safe_dpo = dpo_answer.replace(chr(10), "<br>")

    # Append combined dynamic row data to matrix
    markdown.append(
        f"| {question} "
        f"| {base_answer} "
        f"| {sft_answer} "
        f"| {md_safe_dpo} "
        f"| **DPO Model** "
        f"| Most accurate and domain-specific response |"
    )

# 5. Save final compiled evaluation metrics locally
with open(output_report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(markdown))

print(f"\n🎉 Process complete! Local report saved to: {output_report_path}")
print("-" * 50)

# 6. Push the finalized evaluation report back up to your HF Repository
print(f"📤 Uploading final_evaluation.md to Hugging Face Hub ({OUTPUT_REPO_ID})...")
try:
    api = HfApi(token=HF_TOKEN)
    path_in_repo = "results/final_evaluation.md"

    api.upload_file(
        path_or_fileobj=output_report_path,
        path_in_repo=path_in_repo,
        repo_id=OUTPUT_REPO_ID,
        repo_type="model"  # Change to "dataset" if your repository is a Dataset type
    )
    print(f"🎉 Success! Final report saved in HF repo at: results/final_evaluation.md")

except Exception as e:
    print(f"❌ Hugging Face upload failed. Error details: {e}")

📥 Downloading stage2_evaluation_report.md from Pzazz55/insurance-ai-assistant-finetuning-stage2...
📖 Parsing Stage 2 report data...
📋 Found 10 evaluation cases to complete.
🚀 Starting Evaluation Loop for all 10 questions...

🔄 Processing [1 / 10]
❓ QUESTION:
What mandatory items must be present in a Middle Market risk submission before it can be processed for binding?
----------------------------------------
✅ DPO MODEL ANSWER:
The following mandatory items must be present in a Middle Market risk submission before it can be processed for binding:

1. Policy application: A standard policy application form that includes basic information such as policy type, coverage details, and premium amount.

2. Policy limits: Coverage limits or limits on certain perils (e.g., homeowners coverage up to $500,000). Limits are required because insurers need to assess risk exposure and determine if the policy is affordable.

3. Policy exclusions: Coverage limitations not included in the policy descriptio